# BERT strategy For Author Profiling Task 

Procesamiento de Lenguaje Natural 

Eric Lemus Avalos

En el siguiente Notebook se entrena un modelo BERT a nivel usuario para clasificar el genero y la nacionalidad de los datos de la competencia PAN. La Objetivo es implementar un clasificador mediante un modelo trasnformer BERT para predecir género o nacionalidad por tuit (usando la etiqueta del usuario para todos sus tuits), y para inferir la etiqueta del usuario mediante voto mayoritario. 

## Cargar Data

In [155]:
import os
import re
import json
import time 
from argparse import Namespace
from collections import defaultdict, Counter

import pandas as pd
import numpy as np 

_Funciones para extraer los datos_ 

In [ ]:
def load_truth_values(truth_file):
    """
    Función que lee el archivo truth.txt y extrae los valores de cada usuario en un diccionario.
    """
    truth_dict = {} 
    with open(truth_file, "r", encoding="utf-8") as f:
        for linea in f:
            partes = linea.strip().split(":::") 
            id_usuario, genero, nacionalidad = partes
            truth_dict[id_usuario] = {"genero": genero, "nacionalidad": nacionalidad}
            
    return truth_dict


def extract_tweets_from_xml(xml_file):
    """
    Función que extrae los tweets de un archivo XML.
    """
    tweets = []
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        documents = root.find("documents")
        
        for doc in documents.findall("document"):
            tweet = doc.text
            tweets.append(tweet.strip())
    
    except:
        print(f"error con: {xml_file}")
    
    return tweets

def load_all_tweets(path_train, truth_dict):
    """
    Función que recorre todos los archivos XML en path_train, extrae los tweets y 
    construye un DataFrame con las columnas 'id_usuario', 'tweet', 'genero' y 'nacionalidad'.
    """
    registros = []
    
    for archivo in os.listdir(path_train):
        
        if archivo.endswith(".xml"):
            id_usuario = os.path.splitext(archivo)[0]
            xml_file = os.path.join(path_train, archivo)
            tweets = extract_tweets_from_xml(xml_file)
            genero = truth_dict.get(id_usuario, {}).get("genero")
            nacionalidad = truth_dict.get(id_usuario, {}).get("nacionalidad")
            
            for tweet in tweets:
                registros.append({ "id_usuario": id_usuario,"tweet": tweet,"genero": genero,"nacionalidad": nacionalidad })
                
    df = pd.DataFrame(registros, columns=[ "id_usuario", "tweet", "genero", "nacionalidad" ])
    return df

In [ ]:
paths = Namespace()
paths.train_path = r'../tarea_6/es_train'
paths.train_truth = r'../tarea_6/es_train/truth.txt'
paths.val_path = r'../tarea_6/es_val'
paths.val_truth = r'../tarea_6/es_val/truth.txt'
paths.test_path = r'./competencia-nlp-2025/es_test'
paths.test_truth = r'./competencia-nlp-2025/es_test/truth_order.txt'

In [ ]:
%%time
truth_dict_train = load_truth_values(paths.train_truth)
truth_dict_val = load_truth_values(paths.val_truth)
truth_dict_test = load_truth_values(paths.test_truth)

In [ ]:
%%time 
df_train = load_all_tweets(paths.train_path,truth_dict_train)
df_val = load_all_tweets(paths.val_path,truth_dict_val)
df_test = load_all_tweets(paths.test_path,truth_dict_test)

In [ ]:
orden_ids = list(truth_dict_test.keys())
df_test = df_test.set_index('id_usuario').loc[orden_ids].reset_index()

In [ ]:
from nltk import TweetTokenizer

def tk(tweet): 
    return TweetTokenizer().tokenize(tweet)

def clean_tweets(tweets):
    cleaned_tweets = []
    
    for tweet in tweets:
        tokens = tk(tweet)
        cleaned_tokens = []
        for token in tokens:
            
            if token.startswith('@'):
                cleaned_tokens.append('@usuario')
            elif token.startswith('http'):
                continue  
            else:
                cleaned_tokens.append(token)

        cleaned_tweets.append(' '.join(cleaned_tokens))

    return cleaned_tweets

In [ ]:
%time df_train['tweet_clean'] = clean_tweets(df_train["tweet"].tolist())

In [ ]:
%time df_val['tweet_clean'] = clean_tweets(df_val["tweet"].tolist())

In [ ]:
%time df_test['tweet_clean'] = clean_tweets(df_test["tweet"].tolist())

In [ ]:
output_dir = os.path.join("preprocess_data")
os.makedirs(output_dir, exist_ok=True)
df_train.to_csv(os.path.join(output_dir, "df_train.csv"), index=False)
df_val.to_csv(os.path.join(output_dir, "df_val.csv"), index=False)
df_test.to_csv(os.path.join(output_dir, "df_test.csv"), index=False)

In [89]:
df_train = pd.read_csv("preprocess_data/df_train.csv")
df_val = pd.read_csv("preprocess_data/df_val.csv")
df_test = pd.read_csv("preprocess_data/df_test.csv")

In [90]:
df_train

,id_usuario,tweet_clean,nacionalidad,genero
0,1011b55a13502bcb7562d610a05ef3bd,['#4FRebelionDePatriotas Rebelión Cívico-Milit...,venezuela,male
1,1026000a7c186f3a4aba1053f9956b42,"['Mongodoy es un pobre diablo , un dia nos sor...",peru,male
2,1032e9d0da6d21ceab370759d38930a9,['Que buen orador es Eduardo Torres Dulce . Es...,spain,male
3,10880d19d7346373a42b18e505bbc4d5,['Nueva orden ejecutiva de Trump también afect...,venezuela,female
4,10aef88d048dafbc75639241b9e35df8,['Me gustó un video de @usuario Cómo Hacer Un ...,mexico,female
...,...,...,...,...
3355,ff91e621bd80e9c980a6e7f8550a1d80,"['@usuario Bueno , pero agarra un marcador y e...",venezuela,female
3356,ffb5c29c835e3cecbdaa97bfea5bbe3b,['Que se haga tu voluntad y no la mía Señor .....,colombia,male
3357,ffc9c0b137c5bbb3f9173e7af991e122,['Cómo alguien no se va a enamorar de este ang...,chile,female
3358,ffebe1735cd1f0e69d8210376a9dc377,"['US $ 20 millones #sagrados', '#EducaciónConR...",peru,male


In [91]:
label_map = {'venezuela': 0,  'peru': 1,  'spain': 2, 'mexico': 3, 'colombia': 4,  'argentina': 5,  'chile': 6}
label_map_gen = {'male': 1, 'female': 0}




## BERT nivel usuario y votacion mayoritaria 


In [119]:
import pandas as pd 
df_train = pd.read_csv("./preprocess_data/df_train.csv")
df_val = pd.read_csv("./preprocess_data/df_val.csv")
df_test = pd.read_csv("./preprocess_data/df_test.csv")

In [2]:
df_train.shape

(335998, 5)

In [3]:
df_train.head()

,id_usuario,tweet,genero,nacionalidad,tweet_clean
0,1011b55a13502bcb7562d610a05ef3bd,#4FRebelionDePatriotas Rebelión Cívico-Militar...,male,venezuela,#4FRebelionDePatriotas Rebelión Cívico-Militar...
1,1011b55a13502bcb7562d610a05ef3bd,#4FRebelionDePatriotas ¿Quien sino Chávez?,male,venezuela,#4FRebelionDePatriotas ¿ Quien sino Chávez ?
2,1011b55a13502bcb7562d610a05ef3bd,#4FRebelionDePatriotas Nunca olvidaré aquel “P...,male,venezuela,#4FRebelionDePatriotas Nunca olvidaré aquel “ ...
3,1011b55a13502bcb7562d610a05ef3bd,#4FRebelionDePatriotas La esperanza de un pueb...,male,venezuela,#4FRebelionDePatriotas La esperanza de un pueb...
4,1011b55a13502bcb7562d610a05ef3bd,#4FRebelionDePatriotas Aquel 4F se sintió por ...,male,venezuela,#4FRebelionDePatriotas Aquel 4F se sintió por ...


In [120]:
def transform(dff, label):
    label_map = {'venezuela': 0,  'peru': 1,  'spain': 2, 'mexico': 3, 'colombia': 4,  'argentina': 5,  'chile': 6}
    label_map_gen = {'male': 1, 'female': 0}
    df = dff.copy()
    df['nacionalidad'] = df['nacionalidad'].str.lower().map(label_map)        
    df['genero'] = df['genero'].str.lower().map(label_map_gen)
    return df

In [121]:
dff_train = transform(df_train,label='nacionalidad')
dff_val = transform(df_val,label='nacionalidad')
dff_test = transform(df_test,label='nacionalidad')

In [122]:
def to_low(txt):
    return str(txt).lower()

In [123]:
%%time
dff_val['tweet'] = dff_val['tweet_clean'].apply(to_low)
dff_train['tweet'] = dff_train['tweet_clean'].apply(to_low)
dff_test['tweet'] = dff_test['tweet_clean'].apply(to_low)

CPU times: total: 297 ms
Wall time: 472 ms


Codificar el id del usuario

In [124]:
from sklearn.preprocessing import LabelEncoder

for df in (dff_train, dff_val, dff_test):
    df['id_int'] = LabelEncoder().fit_transform(df['id_usuario'])

In [125]:
dff_train

,id_usuario,tweet,genero,nacionalidad,tweet_clean,id_int
0,1011b55a13502bcb7562d610a05ef3bd,#4frebeliondepatriotas rebelión cívico-militar...,1,0,#4FRebelionDePatriotas Rebelión Cívico-Militar...,0
1,1011b55a13502bcb7562d610a05ef3bd,#4frebeliondepatriotas ¿ quien sino chávez ?,1,0,#4FRebelionDePatriotas ¿ Quien sino Chávez ?,0
2,1011b55a13502bcb7562d610a05ef3bd,#4frebeliondepatriotas nunca olvidaré aquel “ ...,1,0,#4FRebelionDePatriotas Nunca olvidaré aquel “ ...,0
3,1011b55a13502bcb7562d610a05ef3bd,#4frebeliondepatriotas la esperanza de un pueb...,1,0,#4FRebelionDePatriotas La esperanza de un pueb...,0
4,1011b55a13502bcb7562d610a05ef3bd,#4frebeliondepatriotas aquel 4f se sintió por ...,1,0,#4FRebelionDePatriotas Aquel 4F se sintió por ...,0
...,...,...,...,...,...,...
335993,fff46823954870db83b3e6c74a60412c,como sea queria casarse .,1,1,Como sea queria casarse .,3359
335994,fff46823954870db83b3e6c74a60412c,el mundo ya se perdio .,1,1,El mundo ya se perdio .,3359
335995,fff46823954870db83b3e6c74a60412c,le quedo grande el papel de guasón a jared let...,1,1,Le quedo grande el papel de Guasón a Jared Let...,3359
335996,fff46823954870db83b3e6c74a60412c,@usuario una caracteristica tuya mi buen amigo...,1,1,@usuario una caracteristica tuya mi buen amigo...,3359


In [11]:
# sep = "<sep>"
# dff_train["tweet_plus_id"] = dff_train["tweet"] + sep + "id" + sep + dff_train["id_int"].astype(str)
# dff_val["tweet_plus_id"] = dff_val["tweet"] + sep + "id" + sep + dff_val["id_int"].astype(str)
# dff_test["tweet_plus_id"] = dff_test["tweet"] + sep + "id" + sep + dff_test["id_int"].astype(str)

In [126]:
def verificar_tweets_por_usuario(df, esperado=100):
    conteos = df['id_int'].value_counts().reset_index()
    conteos.columns = ['id_int', 'n_tweets']
    usuarios_incompletos = conteos[conteos['n_tweets'] != esperado]
    return usuarios_incompletos['id_int'].to_list()

In [127]:
verificar_tweets_por_usuario(dff_train)

[1684, 1535]

In [128]:
import torch 

CONFIG = {
    "model_name"      : "dccuchile/bert-base-spanish-wwm-uncased",
    "max_len"         : 64,          
    "train_batch_size": 32,
    "test_batch_size" : 64,
    "epochs"          : 3,
    "lr"              : 2e-5,
    "device"          : "cuda" if torch.cuda.is_available() else "cpu",
    "n_models"        : 1,           
    "grad_clip"       : 1.0
}


Tomamos una muestra 

In [16]:
df_train_sample = dff_train[:][:100000]
df_test_sample = dff_test[:][:60000]
df_val_sample =  dff_val[:][:20000]

In [ ]:
def group_by_user(df):
    return df.groupby('id_usuario').agg({ 'tweet': list, 'nacionalidad': 'first', 'genero': 'first', 'id_int': 'first'}).reset_index() 

df_train_grouped, df_val_grouped, df_test_grouped = [group_by_user(df) for df in (dff_train, dff_val, dff_test)]

_PyTorch Dataset_ 

In [21]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, BertTokenizer

class TweetDataset(Dataset):
    def __init__(self, X, y=None, id=None, CONFIG=None):
        self.X = X
        self.y = y
        self.id = id
        self.tk = BertTokenizer.from_pretrained(CONFIG["model_name"],do_lower_case=False)
        
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        output = torch.tensor(self.tk.encode(self.X[idx], max_length=64, pad_to_max_length=True, add_special_tokens=True, truncation_strategy = 'longest_first'))
        if(self.y):
            return {"text": output, "attention":(output!=1).float(), "label":torch.tensor(self.y[idx]), "id": torch.tensor(self.id[idx])}
        else:
            return {"text": output, "attention":(output!=1).float(), "id": torch.tensor(self.id[idx])}

In [136]:
train_dataset = TweetDataset(dff_train['tweet'].to_list(),dff_train['nacionalidad'].to_list(),dff_train['id_int'].to_list(),CONFIG)
test_dataset  = TweetDataset(dff_test['tweet'].to_list(),dff_test['nacionalidad'].to_list(),dff_test['id_int'].to_list(),CONFIG)
val_dataset = TweetDataset(dff_val['tweet'].to_list(),dff_val['nacionalidad'].to_list(),dff_val['id_int'].to_list(),CONFIG)

In [139]:
print(f"Len: {len(train_dataset)}") 
print(f"Elemento idx = 0: {train_dataset[1000]}")

Len: 335998
Elemento idx = 0: {'text': tensor([    4,  1057, 10029,  1470,  1067, 20068,  1041,  2056, 13038,     5,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1]), 'attention': tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]), 'label': tensor(5), 'id': tensor(10)}


C:\Users\ericl\anaconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [140]:
device = torch.device(CONFIG['device'])
lr = CONFIG['lr']
max_grad_norm = 1.0
epochs = CONFIG['epochs']
train_batches = DataLoader(train_dataset, batch_size = CONFIG['train_batch_size'], shuffle = True)
val_batches = DataLoader(val_dataset, batch_size = CONFIG['test_batch_size'], shuffle = False)

In [142]:
batch = next(iter(train_batches))
batch

C:\Users\ericl\anaconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


{'text': tensor([[    4,  1129,  1003,  ...,     1,     1,     1],
         [    4,   985,  8563,  ...,     1,     1,     1],
         [    4,  1645, 15487,  ...,     1,     1,     1],
         ...,
         [    4,  1044,  2670,  ...,     1,     1,     1],
         [    4,  1734,  3172,  ...,     1,     1,     1],
         [    4,  1120,  1067,  ...,     1,     1,     1]]),
 'attention': tensor([[1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         ...,
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.]]),
 'label': tensor([4, 4, 0, 3, 0, 1, 6, 6, 5, 4, 3, 6, 2, 2, 6, 1, 0, 1, 4, 6, 3, 3, 4, 1,
         2, 0, 4, 1, 1, 3, 3, 6]),
 'id': tensor([2984,  294, 2913, 1451, 2560, 1960, 1427,  490, 2924, 1422,  224, 1405,
         2580, 2239, 2254,   92, 2672,  264, 1094, 1771,   44, 3166, 3097, 2857,
         3349,  943, 1325, 3280,  401,  691, 1428, 141

_Modelo_ 

In [143]:
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from torch.optim import AdamW  

In [147]:
class BertModel(nn.Module):
    def __init__(self, model_name: str, num_labels: int):
        super(BertModel, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(0.1)
        self.clf  = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, x, att):
        pooled = self.bert(x, attention_mask=att)[1]
        logits = self.clf(self.drop(pooled))
        return logits


In [148]:
model = BertModel(CONFIG['model_name'], num_labels=7).to(device)
optimizer = AdamW(model.parameters(), lr=lr)
total_steps = len(train_batches) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = 0, num_training_steps = total_steps)
criterion = nn.CrossEntropyLoss()

Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [149]:
# model 

In [151]:
# labels_dict = dict(zip(df_train_grouped["id_int"], df_train_grouped["nacionalidad"]))    
# labels_dict_val = dict(zip(df_val_grouped["id_int"], df_val_grouped["nacionalidad"]))
# # labels_dict_test = dict(zip(df_test_grouped["id_int"], df_test_grouped["nacionalidad"]))
# results  = defaultdict(list)                          
# evaluated_users, user_hits, user_total = set(), 0, 0

_Entrenamiento_

In [210]:
import torch
import torch.nn.functional as F
from collections import defaultdict, Counter

def test_model(model, val_batches, config, true_labels_val, ids_con_99):
    
    device = torch.device(config['device'])
    model.eval()
    resultados_val = defaultdict(list)
    
    with torch.no_grad():
        for batch in val_batches:
            x   = batch['text'].to(device)
            att = batch['attention'].to(device)
            uid = batch['id']          
            logits = model(x, att)
            probs  = F.softmax(logits, dim=1)
            preds  = probs.argmax(dim=1).cpu().tolist()
            uids   = uid.cpu().tolist()

            for u, p in zip(uids, preds):
                resultados_val[u].append(p)

    aciertos = 0
    total   = 0
    for u, lst in resultados_val.items():
        esperado = 100
        if len(lst) == esperado:
            voto, _ = Counter(lst).most_common(1)[0]
            if true_labels_val[u] == voto:
                aciertos += 1
            total += 1
    acc_usuario = aciertos / total if total else 0.0
    return acc_usuario


In [152]:
true_labels = dict(zip(df_train_grouped["id_int"],df_train_grouped["nacionalidad"]))
true_labels_val = labels_dict_val = dict(zip(df_val_grouped["id_int"], df_val_grouped["nacionalidad"]))

In [154]:
k = 1
ids_con_99 = (1684, 1535)

for epoch in range(CONFIG['epochs']):
    model.train()
    resultados = defaultdict(list)

    for batch in tqdm(train_batches):
        input_ids      = batch["text"].to(device)          
        attention_mask = batch["attention"].to(device)      
        labels         = batch["label"].to(device)         
        uids_tensor    = batch["id"]                    
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)          
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        preds = torch.argmax(logits, dim=1).cpu().tolist()
        uids  = uids_tensor.cpu().tolist()
        for uid, p in zip(uids, preds):
            resultados[uid].append(p)

    final_preds = {}
    for uid, preds_list in resultados.items():
        esperado = 99 if uid in ids_con_99 else 100
        if len(preds_list) == esperado:
            voto, _ = Counter(preds_list).most_common(1)[0]
            final_preds[uid] = voto
    #     else:
    #         print(f"[Época {epoch+1}] Usuario {uid} tiene {len(preds_list)} tuits (esperado {esperado}) – se omite")

    aciertos = sum(1 for uid, pred in final_preds.items() if true_labels.get(uid) == pred)
    total_eval = len(final_preds)
    acc_usuario = aciertos / total_eval if total_eval else 0.0
    print(f"Época {epoch+1:02d} → Accuracy por usuario: {acc_usuario:.4f} " f"({aciertos}/{total_eval})")

    acc_val = test_model(model,val_batches,CONFIG, true_labels_val,  ids_con_99)
    print(f"Época {epoch+1:02d} → Val Accuracy por usuario: {acc_val:.4f}")

torch.save(model.state_dict(), 'Models/model_'+str(k)+'.pt')

  0%|          | 0/10500 [00:00<?, ?it/s]

C:\Users\ericl\anaconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Época 01 → Accuracy por usuario: 0.8193 (2753/3360)
Época 01 → Val Accuracy por usuario: 0.8036


  0%|          | 0/10500 [00:00<?, ?it/s]

Época 02 → Accuracy por usuario: 0.9455 (3177/3360)
Época 02 → Val Accuracy por usuario: 0.8524


  0%|          | 0/10500 [00:00<?, ?it/s]

Época 03 → Accuracy por usuario: 0.9792 (3290/3360)
Época 03 → Val Accuracy por usuario: 0.8869


In [157]:
k = 'nacionalidad'
os.makedirs("Models", exist_ok=True)
model_path = os.path.join("Models", f"model_{str(k)}.pt")
torch.save(model.state_dict(), model_path)

In [169]:
test_batches = DataLoader(test_dataset, batch_size = CONFIG['test_batch_size'], shuffle = False)

In [173]:
len(test_batches)*64//100

2800

In [160]:
model.eval()
resultados_test = defaultdict(list)
with torch.no_grad():
    for batch in tqdm(test_batches):
        input_ids      = batch["text"].to(device)
        attention_mask = batch["attention"].to(device)
        uids           = batch["id"]            
        logits = model(input_ids, attention_mask)    
        preds  = logits.argmax(dim=1).cpu().tolist()
        for uid, p in zip(uids, preds):
            resultados_test[uid].append(p)

  0%|          | 0/4375 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
C:\Users\ericl\anaconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [176]:
def majority_vote_dict(raw_results, ids_con_99=None):
    clean = defaultdict(list)
    for uid_tensor, preds in raw_results.items():
        uid = uid_tensor.item() if hasattr(uid_tensor, "item") else int(uid_tensor)
        for p in preds:
            clean[uid].append(p.item() if hasattr(p, "item") else int(p))

    final_preds = {}
    esperado = 100
    for uid, pred_list in clean.items():
        if len(pred_list) >= esperado:
            voto, _ = Counter(pred_list).most_common(1)[0]
            final_preds[uid] = voto
        else:
            print(f"Usuario {uid}: sólo {len(pred_list)} tuits (esperado {esperado}) → se omite")

    return final_preds



In [182]:
# final_preds_labels

In [167]:
mapping = dict(zip(dff_test["id_int"],dff_test["id_usuario"]))
final_preds_original = { mapping[id_int]: pred for id_int, pred in final_preds_test.items() if id_int in mapping}

In [195]:
label_map = { 'venezuela': 0,  'peru': 1,  'spain': 2, 'mexico': 3, 'colombia': 4,  'argentina': 5,  'chile': 6 }
inverse_label_map = {code: country for country, code in label_map.items()}
final_preds_test = majority_vote_dict(resultados_test, ids_con_99)
mapping = dict(zip(dff_test["id_int"],dff_test["id_usuario"]))
final_preds_original = { mapping[id_int]: pred for id_int, pred in final_preds_test.items() if id_int in mapping}
final_preds_original = { user: inverse_label_map[pred] for user, pred in final_preds_original.items() }
df_output = pd.DataFrame(list(final_preds_original.items()),columns=['id_usuario', 'nacionalidad_predicha'])
df_output.to_csv("predicciones_nacionalidad.csv", index=False)

___Para el genero___ 

In [196]:
df_train = pd.read_csv("./preprocess_data/df_train.csv")
df_val = pd.read_csv("./preprocess_data/df_val.csv")
df_test = pd.read_csv("./preprocess_data/df_test.csv")

In [197]:
dff_train = transform(df_train,label='genero')
dff_val = transform(df_val,label='genero')
dff_test = transform(df_test,label='genero')

In [198]:
dff_val['tweet'] = dff_val['tweet_clean'].apply(to_low)
dff_train['tweet'] = dff_train['tweet_clean'].apply(to_low)
dff_test['tweet'] = dff_test['tweet_clean'].apply(to_low)

In [199]:
for df in (dff_train, dff_val, dff_test):
    df['id_int'] = LabelEncoder().fit_transform(df['id_usuario'])

In [200]:
def group_by_user(df):
    return df.groupby('id_usuario').agg({ 'tweet': list, 'nacionalidad': 'first', 'genero': 'first', 'id_int': 'first'}).reset_index() 

df_train_grouped, df_val_grouped, df_test_grouped = [group_by_user(df) for df in (dff_train, dff_val, dff_test)]

In [207]:
train_dataset = TweetDataset(dff_train['tweet'].to_list(),dff_train['genero'].to_list(),dff_train['id_int'].to_list(),CONFIG)
test_dataset  = TweetDataset(dff_test['tweet'].to_list(),dff_test['genero'].to_list(),dff_test['id_int'].to_list(),CONFIG)
val_dataset = TweetDataset(dff_val['tweet'].to_list(),dff_val['genero'].to_list(),dff_val['id_int'].to_list(),CONFIG)

In [208]:
train_batches = DataLoader(train_dataset, batch_size = CONFIG['train_batch_size'], shuffle = True)
val_batches = DataLoader(val_dataset, batch_size = CONFIG['test_batch_size'], shuffle = False)

In [209]:
model_genro = BertModel(CONFIG['model_name'], num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=lr)
total_steps = len(train_batches) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = 0, num_training_steps = total_steps)
criterion = nn.CrossEntropyLoss()

Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [212]:
true_labels = dict(zip(df_train_grouped["id_int"],df_train_grouped["genero"]))
true_labels_val = labels_dict_val = dict(zip(df_val_grouped["id_int"], df_val_grouped["genero"]))

In [214]:
k = 'modelo'
for epoch in range(CONFIG['epochs']):
    model.train()
    resultados = defaultdict(list)

    for batch in tqdm(train_batches):
        input_ids      = batch["text"].to(device)          
        attention_mask = batch["attention"].to(device)      
        labels         = batch["label"].to(device)         
        uids_tensor    = batch["id"]                    
        optimizer.zero_grad()
        logits = model_genro(input_ids, attention_mask)          
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        preds = torch.argmax(logits, dim=1).cpu().tolist()
        uids  = uids_tensor.cpu().tolist()
        for uid, p in zip(uids, preds):
            resultados[uid].append(p)

    final_preds = {}
    for uid, preds_list in resultados.items():
        esperado = 99 if uid in ids_con_99 else 100
        if len(preds_list) == esperado:
            voto, _ = Counter(preds_list).most_common(1)[0]
            final_preds[uid] = voto
    
    aciertos = sum(1 for uid, pred in final_preds.items() if true_labels.get(uid) == pred)
    total_eval = len(final_preds)
    acc_usuario = aciertos / total_eval if total_eval else 0.0
    acc_val = test_model(model_genro,val_batches,CONFIG, true_labels_val,  ids_con_99)
    print(f"Época {epoch+1:02d} → Accuracy por usuario: {acc_usuario:.4f} " f"({aciertos}/{total_eval})")
    print(f"Época {epoch+1:02d} → Val Accuracy por usuario: {acc_val:.4f}")

torch.save(model_genro.state_dict(), 'Models/model_'+k+'.pt')

  0%|          | 0/10500 [00:00<?, ?it/s]

C:\Users\ericl\anaconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Época 01 → Accuracy por usuario: 0.4967 (1669/3360)
Época 01 → Val Accuracy por usuario: 0.4917


  0%|          | 0/10500 [00:00<?, ?it/s]

Época 02 → Accuracy por usuario: 0.4958 (1666/3360)
Época 02 → Val Accuracy por usuario: 0.4917


  0%|          | 0/10500 [00:00<?, ?it/s]

Época 03 → Accuracy por usuario: 0.4961 (1667/3360)
Época 03 → Val Accuracy por usuario: 0.4917


In [215]:
k = 'genero'
os.makedirs("Models", exist_ok=True)
model_path = os.path.join("Models", f"model_{str(k)}.pt")
torch.save(model_genro.state_dict(), model_path)

In [216]:
test_batches = DataLoader(test_dataset, batch_size = CONFIG['test_batch_size'], shuffle = False)

In [217]:
model_genro.eval()
resultados_test = defaultdict(list)
with torch.no_grad():
    for batch in tqdm(test_batches):
        input_ids      = batch["text"].to(device)
        attention_mask = batch["attention"].to(device)
        uids           = batch["id"]            
        logits = model_genro(input_ids, attention_mask)    
        preds  = logits.argmax(dim=1).cpu().tolist()
        for uid, p in zip(uids, preds):
            resultados_test[uid].append(p)

  0%|          | 0/4375 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
C:\Users\ericl\anaconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [220]:
mapping = dict(zip(dff_test["id_int"],dff_test["id_usuario"]))
final_preds_original = { mapping[id_int]: pred for id_int, pred in final_preds_test.items() if id_int in mapping}

In [221]:
label_map = { 'male': 1, 'female': 0 }
inverse_label_map = {code: gen for gen, code in label_map.items()}
final_preds_test = majority_vote_dict(resultados_test, ids_con_99)
mapping = dict(zip(dff_test["id_int"],dff_test["id_usuario"]))
final_preds_original = { mapping[id_int]: pred for id_int, pred in final_preds_test.items() if id_int in mapping}
final_preds_original = { user: inverse_label_map[pred] for user, pred in final_preds_original.items() }
df_output = pd.DataFrame(list(final_preds_original.items()),columns=['id_usuario', 'genero'])
df_output.to_csv("predicciones_genero.csv", index=False)

In [222]:
with open("pred_gen_nat.csv", "w") as f, open("predicciones_genero.csv", "r") as g, open("predicciones_nacionalidad.csv", "r") as n:
    f.write('Id,Gender:::Nationality\n')
    cont_gen = g.read().split()
    cont_nat = n.read().split()
    assert (len(cont_gen) == len(cont_nat))
    
    for i, label in enumerate(cont_gen):
        if i > 0:
            ID, gen = label.split(',')[0], label.split(',')[1]
            nat = cont_nat[i].split(',')[1]
            f.write(ID + ',' + gen + ':::' + nat + '\n')